# TRACE — Image & Video Processing Lab

This notebook demonstrates **all classical IVP techniques** applied to industrial manufacturing frames from the TRACE corpus.

**Techniques covered:**
1. Color Space Conversion (RGB → Grayscale, HSV, LAB)
2. Histogram Analysis & CLAHE Equalization
3. Image Filtering & Convolution (Gaussian Blur, Sharpening, Emboss, Laplacian)
4. Edge Detection (Sobel X/Y/Magnitude/Direction + Canny)
5. Morphological Operations (Erosion, Dilation, Opening, Closing, Gradient)
6. Feature Detection — SIFT & ORB Keypoints
7. Optical Flow — Sparse Lucas-Kanade + Dense Farneback
8. HOG — Histogram of Oriented Gradients
9. Keyframe Extraction from Video

## Setup — Install dependencies

In [ ]:
# Install if not already present
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install',
                'opencv-python-headless', 'numpy', 'matplotlib', 'ipython'], check=True)
print('✅ Dependencies ready')

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import math, os, sys

# Point to TRACE backend so we can import the ivp module
TRACE_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
sys.path.insert(0, os.path.join(TRACE_ROOT, 'backend'))

%matplotlib inline
plt.rcParams['figure.facecolor'] = '#0f172a'
plt.rcParams['axes.facecolor']   = '#1e293b'
plt.rcParams['axes.labelcolor']  = '#94a3b8'
plt.rcParams['xtick.color']      = '#64748b'
plt.rcParams['ytick.color']      = '#64748b'
plt.rcParams['text.color']       = '#e2e8f0'
plt.rcParams['figure.dpi']       = 110

print(f'OpenCV  : {cv2.__version__}')
print(f'NumPy   : {np.__version__}')

## Helper — Synthetic Industrial Frame (works without video files)

In [ ]:
def make_industrial_frame(w=640, h=480, seed=42):
    """Generates a synthetic assembly workpiece frame for demonstration."""
    np.random.seed(seed)
    frame = np.zeros((h, w, 3), dtype=np.uint8)
    for y in range(h):
        v = int(20 + (y/h)*40)
        frame[y, :] = (v, v+5, v+2)
    # Metal workpiece
    cv2.rectangle(frame, (80, 100), (560, 380), (120, 115, 110), -1)
    noise = np.random.randint(0, 20, (280, 480, 3), dtype=np.uint8)
    frame[100:380, 80:560] = cv2.add(frame[100:380, 80:560], noise)
    # Bolt holes
    for cx, cy in [(160,180),(480,180),(160,300),(480,300),(320,240)]:
        cv2.circle(frame, (cx,cy), 18, (30,28,25), -1)
        cv2.circle(frame, (cx,cy), 14, (20,18,16), -1)
        cv2.circle(frame, (cx,cy),  6, (60,55,50), -1)
    # T-slot grid
    for y in [140,200,260,320]: cv2.line(frame,(80,y),(560,y),(170,165,160),2)
    for x in [160,240,320,400,480]: cv2.line(frame,(x,100),(x,380),(170,165,160),1)
    # Fasteners
    for cx,cy in [(200,160),(440,160),(200,320),(440,320)]:
        cv2.circle(frame,(cx,cy),10,(200,190,180),-1)
        cv2.circle(frame,(cx,cy), 5,(230,225,220),-1)
    cv2.putText(frame,'TRACE IVP — Industrial Assembly Frame',(90,55),
                cv2.FONT_HERSHEY_SIMPLEX,0.65,(200,200,200),2)
    return frame

# Load a real image if available, else use synthetic
IMAGE_PATH = None  # Set to 'path/to/image.jpg' if you have one
VIDEO_PATH = None  # Set to 'path/to/video.mp4' if you have one

if IMAGE_PATH and os.path.exists(IMAGE_PATH):
    bgr = cv2.resize(cv2.imread(IMAGE_PATH), (640, 480))
    print(f'Loaded image: {IMAGE_PATH}')
else:
    bgr = make_industrial_frame()
    print('Using synthetic industrial frame')

rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
plt.figure(figsize=(8,5))
plt.imshow(rgb); plt.axis('off'); plt.title('Source Frame', color='white', fontsize=13)
plt.tight_layout(); plt.show()

---
## 1. Color Space Conversion

In [ ]:
gray = cv2.cvtColor(bgr, cv2.COLOR_BGR2GRAY)
hsv  = cv2.cvtColor(bgr, cv2.COLOR_BGR2HSV)
lab  = cv2.cvtColor(bgr, cv2.COLOR_BGR2LAB)

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
titles = ['Original RGB', 'Grayscale', 'HSV', 'LAB']
imgs   = [cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB), gray, hsv, lab]
cmaps  = [None, 'gray', None, None]
for ax, img, t, cm in zip(axes, imgs, titles, cmaps):
    ax.imshow(img if cm is None else img, cmap=cm)
    ax.set_title(t, color='#60a5fa'); ax.axis('off')
fig.suptitle('Color Space Conversion', color='white', fontsize=14, y=1.01)
plt.tight_layout(); plt.show()

h,s,v = cv2.split(hsv)
print(f'Mean brightness (gray): {gray.mean():.1f}')
print(f'Mean hue (HSV):         {h.mean():.1f}  |  Mean saturation: {s.mean():.1f}')
print(f'Std deviation (gray):   {gray.std():.1f}')

---
## 2. Histogram Analysis & CLAHE Equalization

In [ ]:
# Channel histograms
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
colors_bgr = [('Blue','#3b82f6',0),('Green','#22c55e',1),('Red','#ef4444',2)]
for ax,(label,color,ch) in zip(axes, colors_bgr):
    hist = cv2.calcHist([bgr],[ch],None,[256],[0,256]).flatten()
    ax.fill_between(range(256), hist, alpha=0.5, color=color)
    ax.plot(hist, color=color, lw=1.5)
    ax.set_title(f'{label} Channel Histogram', color='#60a5fa')
    ax.set_xlabel('Intensity'); ax.set_ylabel('Count')
plt.tight_layout(); plt.show()

# CLAHE equalization
lab2 = cv2.cvtColor(bgr, cv2.COLOR_BGR2LAB)
l,a,b = cv2.split(lab2)
clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))
l_eq  = clahe.apply(l)
bgr_eq = cv2.cvtColor(cv2.merge([l_eq,a,b]), cv2.COLOR_LAB2BGR)
gray_eq = cv2.equalizeHist(gray)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, img, t in zip(axes,
    [cv2.cvtColor(bgr,cv2.COLOR_BGR2RGB), cv2.cvtColor(bgr_eq,cv2.COLOR_BGR2RGB), gray_eq],
    ['Original','CLAHE Equalized (LAB)','Global Hist Equalized (Gray)']):
    ax.imshow(img, cmap='gray' if img.ndim==2 else None); ax.set_title(t,color='#60a5fa'); ax.axis('off')
plt.tight_layout(); plt.show()
print(f'Original contrast std:  {gray.std():.2f}')
print(f'Equalized contrast std: {gray_eq.std():.2f}')

---
## 3. Image Filtering & Convolution

In [ ]:
# Convolution kernels
K_sharp  = np.array([[ 0,-1, 0],[-1, 5,-1],[ 0,-1, 0]], np.float32)
K_emboss = np.array([[-2,-1, 0],[-1, 1, 1],[ 0, 1, 2]], np.float32)

blur3    = cv2.GaussianBlur(gray,(3,3),0)
blur7    = cv2.GaussianBlur(gray,(7,7),0)
blur15   = cv2.GaussianBlur(gray,(15,15),0)
sharp    = np.clip(cv2.filter2D(gray,-1,K_sharp),0,255).astype(np.uint8)
emboss   = np.clip(cv2.filter2D(gray,-1,K_emboss)+128,0,255).astype(np.uint8)
laplacian= np.abs(cv2.Laplacian(gray,cv2.CV_64F)).astype(np.uint8)

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
pairs = [
    (gray,'Original Gray'),  (blur3,'Gaussian σ=3'),
    (blur7,'Gaussian σ=7'),  (blur15,'Gaussian σ=15'),
    (sharp,'Sharpened'),     (emboss,'Emboss'),
    (laplacian,'Laplacian'), (None, None)
]
for ax,(img,t) in zip(axes.flat, pairs):
    if img is None: ax.axis('off'); continue
    ax.imshow(img, cmap='gray'); ax.set_title(t,color='#60a5fa'); ax.axis('off')
fig.suptitle('Convolution & Filtering', color='white', fontsize=14)
plt.tight_layout(); plt.show()

print(f'Original sharpness (Laplacian var): {cv2.Laplacian(gray,cv2.CV_64F).var():.1f}')
print(f'Blurred  sharpness (Laplacian var): {cv2.Laplacian(blur7,cv2.CV_64F).var():.1f}')
print(f'Sharpened sharpness (Laplacian var):{cv2.Laplacian(sharp,cv2.CV_64F).var():.1f}')

---
## 4. Edge Detection — Sobel & Canny

In [ ]:
blurred = cv2.GaussianBlur(gray,(5,5),0)

Gx  = cv2.Sobel(blurred,cv2.CV_64F,1,0,ksize=3)
Gy  = cv2.Sobel(blurred,cv2.CV_64F,0,1,ksize=3)
mag = np.sqrt(Gx**2+Gy**2)
ang = np.arctan2(Gy,Gx)*180/math.pi

Gx_vis  = np.clip(np.abs(Gx),0,255).astype(np.uint8)
Gy_vis  = np.clip(np.abs(Gy),0,255).astype(np.uint8)
mag_vis = ((mag/mag.max())*255).astype(np.uint8) if mag.max()>0 else np.zeros_like(gray)
ang_vis = ((ang+180)/360*255).astype(np.uint8)

canny_tight  = cv2.Canny(blurred,100,200)
canny_medium = cv2.Canny(blurred,50,150)
canny_loose  = cv2.Canny(blurred,20,80)

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
pairs = [
    (gray,'Original Gray'),(Gx_vis,'Sobel Gx (horizontal)'),
    (Gy_vis,'Sobel Gy (vertical)'),(mag_vis,'Gradient Magnitude |G|'),
    (ang_vis,'Gradient Direction θ'),(canny_tight,'Canny tight (100,200)'),
    (canny_medium,'Canny medium (50,150)'),(canny_loose,'Canny loose (20,80)'),
]
for ax,(img,t) in zip(axes.flat,pairs):
    ax.imshow(img,cmap='gray'); ax.set_title(t,color='#60a5fa'); ax.axis('off')
fig.suptitle('Edge Detection — Sobel + Canny', color='white', fontsize=14)
plt.tight_layout(); plt.show()

edge_density = np.count_nonzero(canny_medium)/(gray.shape[0]*gray.shape[1])
print(f'Mean gradient magnitude:  {mag.mean():.2f}')
print(f'Edge density (medium):    {edge_density*100:.2f}%')
print(f'Canny edges — tight: {np.count_nonzero(canny_tight)} | medium: {np.count_nonzero(canny_medium)} | loose: {np.count_nonzero(canny_loose)}')

---
## 5. Morphological Operations

In [ ]:
_, binary = cv2.threshold(cv2.GaussianBlur(gray,(5,5),0), 0, 255,
                          cv2.THRESH_BINARY+cv2.THRESH_OTSU)

k3 = cv2.getStructuringElement(cv2.MORPH_ELLIPSE,(3,3))
k7 = cv2.getStructuringElement(cv2.MORPH_ELLIPSE,(7,7))

eroded   = cv2.erode(binary,k3,iterations=2)
dilated  = cv2.dilate(binary,k3,iterations=2)
opened   = cv2.morphologyEx(binary,cv2.MORPH_OPEN,k7)
closed   = cv2.morphologyEx(binary,cv2.MORPH_CLOSE,k7)
gradient = cv2.morphologyEx(binary,cv2.MORPH_GRADIENT,k3)
tophat   = cv2.morphologyEx(binary,cv2.MORPH_TOPHAT,k7)
blackhat = cv2.morphologyEx(binary,cv2.MORPH_BLACKHAT,k7)

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
pairs = [
    (gray,'Gray Input'),(binary,'Binary (Otsu)'),
    (eroded,'Erosion'),(dilated,'Dilation'),
    (opened,'Opening'),(closed,'Closing'),
    (gradient,'Morph Gradient'),(tophat,'Top-Hat'),
]
for ax,(img,t) in zip(axes.flat,pairs):
    ax.imshow(img,cmap='gray'); ax.set_title(t,color='#60a5fa'); ax.axis('off')
fig.suptitle('Morphological Operations', color='white', fontsize=14)
plt.tight_layout(); plt.show()

for name,img in [('binary',binary),('eroded',eroded),('dilated',dilated)]:
    pct = np.count_nonzero(img)/img.size*100
    print(f'{name:10s} fill: {pct:.1f}%')

---
## 6. Feature Detection — SIFT & ORB

In [ ]:
sift = cv2.SIFT_create(nfeatures=500)
kp_s, ds = sift.detectAndCompute(gray, None)
img_sift = cv2.drawKeypoints(bgr.copy(), kp_s, None,
                             flags=cv2.DRAW_MATCHES_FLAGS_DRAW_RICH_KEYPOINTS)

orb = cv2.ORB_create(nfeatures=500)
kp_o, do = orb.detectAndCompute(gray, None)
img_orb = cv2.drawKeypoints(bgr.copy(), kp_o, None,
                            flags=cv2.DRAW_MATCHES_FLAGS_DRAW_RICH_KEYPOINTS)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].imshow(cv2.cvtColor(img_sift,cv2.COLOR_BGR2RGB))
axes[0].set_title(f'SIFT — {len(kp_s)} keypoints  |  descriptor: 128-d float',color='#60a5fa'); axes[0].axis('off')
axes[1].imshow(cv2.cvtColor(img_orb,cv2.COLOR_BGR2RGB))
axes[1].set_title(f'ORB — {len(kp_o)} keypoints  |  descriptor: 32-byte binary',color='#60a5fa'); axes[1].axis('off')
fig.suptitle('Feature Detection', color='white', fontsize=14)
plt.tight_layout(); plt.show()

if kp_s:
    sizes = [k.size for k in kp_s]; responses = [k.response for k in kp_s]
    print(f'SIFT: {len(kp_s)} keypoints | mean size: {np.mean(sizes):.1f} | mean response: {np.mean(responses):.4f}')
    print(f'ORB:  {len(kp_o)} keypoints | descriptor shape: {do.shape if do is not None else "none"}')

---
## 7. Optical Flow — Lucas-Kanade (Sparse) & Farneback (Dense)

In [ ]:
# Simulate motion: shift frame by 5px right, 3px down
M = np.float32([[1,0,5],[0,1,3]])
bgr2 = cv2.warpAffine(bgr, M, (bgr.shape[1], bgr.shape[0]))
gray2 = cv2.cvtColor(bgr2, cv2.COLOR_BGR2GRAY)

# --- Sparse Lucas-Kanade ---
corners = cv2.goodFeaturesToTrack(gray, 200, 0.01, 10)
lk_params = dict(winSize=(15,15), maxLevel=3,
                 criteria=(cv2.TERM_CRITERIA_EPS|cv2.TERM_CRITERIA_COUNT,30,0.01))
p1, st, _ = cv2.calcOpticalFlowPyrLK(gray, gray2, corners, None, **lk_params)
good_new = p1[st==1]; good_old = corners[st==1]

lk_vis = bgr2.copy()
for new,old in zip(good_new, good_old):
    a,b = int(new[0]),int(new[1]); c,d = int(old[0]),int(old[1])
    cv2.arrowedLine(lk_vis,(c,d),(a,b),(0,255,0),1,tipLength=0.3)
    cv2.circle(lk_vis,(a,b),3,(0,0,255),-1)

# --- Dense Farneback ---
flow = cv2.calcOpticalFlowFarneback(gray, gray2, None, 0.5,3,15,3,5,1.2,0)
mag,ang = cv2.cartToPolar(flow[...,0], flow[...,1])
hsv_flow = np.zeros((*gray.shape,3),dtype=np.uint8)
hsv_flow[...,0] = ang*180/math.pi/2
hsv_flow[...,1] = 255
hsv_flow[...,2] = cv2.normalize(mag,None,0,255,cv2.NORM_MINMAX)
dense_vis = cv2.cvtColor(hsv_flow, cv2.COLOR_HSV2BGR)

fig, axes = plt.subplots(1, 4, figsize=(18, 4))
for ax,img,t in zip(axes,
    [cv2.cvtColor(bgr,cv2.COLOR_BGR2RGB), cv2.cvtColor(bgr2,cv2.COLOR_BGR2RGB),
     cv2.cvtColor(lk_vis,cv2.COLOR_BGR2RGB), cv2.cvtColor(dense_vis,cv2.COLOR_BGR2RGB)],
    ['Frame N','Frame N+1','Lucas-Kanade Sparse Flow','Farneback Dense Flow (HSV)']):
    ax.imshow(img); ax.set_title(t,color='#60a5fa'); ax.axis('off')
fig.suptitle('Optical Flow', color='white', fontsize=14)
plt.tight_layout(); plt.show()

print(f'Tracked points:        {len(good_new)}')
print(f'Mean flow magnitude:   {mag.mean():.4f} px')
print(f'Max flow magnitude:    {mag.max():.4f} px')
print(f'Motion active:         {mag.mean() > 1.0}')

---
## 8. HOG — Histogram of Oriented Gradients

In [ ]:
resized = cv2.resize(bgr,(64,128))
gray_r  = cv2.cvtColor(resized, cv2.COLOR_BGR2GRAY)

hog = cv2.HOGDescriptor((64,128),(16,16),(8,8),(8,8),9)
descriptor = hog.compute(gray_r).flatten()

# Visualize gradient arrows per cell
hog_vis = resized.copy()
for r in range(16):
    for c in range(8):
        cy_c,cx_c = r*8+4, c*8+4
        cell = gray_r[r*8:(r+1)*8, c*8:(c+1)*8]
        gx = cv2.Sobel(cell,cv2.CV_64F,1,0,ksize=3).mean()
        gy = cv2.Sobel(cell,cv2.CV_64F,0,1,ksize=3).mean()
        angle = math.atan2(gy,gx)
        m = min(math.hypot(gx,gy), 3)
        ex,ey = int(cx_c+m*math.cos(angle)), int(cy_c+m*math.sin(angle))
        cv2.arrowedLine(hog_vis,(cx_c,cy_c),(ex,ey),(0,255,255),1,tipLength=0.4)

fig, axes = plt.subplots(1, 3, figsize=(14, 5))
axes[0].imshow(cv2.cvtColor(resized,cv2.COLOR_BGR2RGB)); axes[0].set_title('Resized (64×128)',color='#60a5fa'); axes[0].axis('off')
axes[1].imshow(cv2.cvtColor(hog_vis,cv2.COLOR_BGR2RGB)); axes[1].set_title('HOG Cell Gradients',color='#60a5fa'); axes[1].axis('off')
axes[2].plot(descriptor[:200],'#38bdf8',lw=0.8); axes[2].set_facecolor('#1e293b')
axes[2].set_title(f'HOG Descriptor (first 200 of {len(descriptor)})',color='#60a5fa')
axes[2].set_xlabel('Bin'); axes[2].set_ylabel('Magnitude')
fig.suptitle('HOG — Histogram of Oriented Gradients', color='white', fontsize=14)
plt.tight_layout(); plt.show()

print(f'Descriptor length:   {len(descriptor)}')
print(f'Mean:   {descriptor.mean():.6f} | Std: {descriptor.std():.6f} | Max: {descriptor.max():.4f}')
print(f'Non-zero bins: {np.count_nonzero(descriptor)} / {len(descriptor)}')

---
## 9. Keyframe Extraction from Video

In [ ]:
# If you have a video file, set VIDEO_PATH above.
# This cell simulates keyframe extraction from a synthetic video stream.

if VIDEO_PATH and os.path.exists(VIDEO_PATH):
    sys.path.insert(0, os.path.join(TRACE_ROOT, 'backend'))
    from app.ivp.processing import extract_keyframes
    result = extract_keyframes(VIDEO_PATH, diff_threshold=15.0, max_frames=16)
    kfs = result['keyframes']
    print(f'Extracted {len(kfs)} keyframes from {result["total_frames"]} total frames @ {result["fps"]:.1f} fps')
    
    n = min(len(kfs), 8)
    fig, axes = plt.subplots(2, 4, figsize=(16, 8))
    for ax, kf in zip(axes.flat, kfs[:n]):
        import base64
        img_bytes = base64.b64decode(kf['image_png_b64'])
        img_arr = np.frombuffer(img_bytes, np.uint8)
        img_bgr = cv2.imdecode(img_arr, cv2.IMREAD_COLOR)
        ax.imshow(cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB))
        ax.set_title(f't={kf["timestamp_s"]}s | Δ={kf["inter_frame_diff"]}', color='#60a5fa', fontsize=8)
        ax.axis('off')
    for ax in axes.flat[n:]: ax.axis('off')
    plt.tight_layout(); plt.show()
else:
    print('Simulating keyframe extraction using inter-frame difference logic...')
    # Simulate by applying different transformations to the synthetic frame
    frames = []
    for i in range(30):
        M = np.float32([[1,0,i*2],[0,1,i]])
        f = cv2.warpAffine(bgr, M, (bgr.shape[1],bgr.shape[0]))
        # Every 7th frame: add a big change (simulate scene cut)
        if i % 7 == 0:
            f = cv2.flip(f, 1)
        frames.append(f)

    keyframes_sim = []
    prev = cv2.cvtColor(frames[0], cv2.COLOR_BGR2GRAY)
    for idx, frame in enumerate(frames[1:], 1):
        curr = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        diff = float(cv2.absdiff(curr, prev).mean())
        if diff > 8.0:
            edge_map = cv2.Canny(cv2.GaussianBlur(curr,(5,5),0), 50, 150)
            keyframes_sim.append({'idx':idx,'diff':round(diff,2),
                                  'edge_density':round(np.count_nonzero(edge_map)/edge_map.size*100,2),
                                  'sharpness':round(cv2.Laplacian(curr,cv2.CV_64F).var(),1),
                                  'frame':frame})
        prev = curr

    print(f'Keyframes extracted: {len(keyframes_sim)} from {len(frames)} simulated frames')
    n = min(len(keyframes_sim), 8)
    if n > 0:
        fig, axes = plt.subplots(2, 4, figsize=(16, 8))
        for ax, kf in zip(axes.flat, keyframes_sim[:n]):
            ax.imshow(cv2.cvtColor(kf['frame'],cv2.COLOR_BGR2RGB))
            ax.set_title(f'frame {kf["idx"]} | Δ={kf["diff"]} | edge={kf["edge_density"]}%',
                         color='#60a5fa', fontsize=8); ax.axis('off')
        for ax in axes.flat[n:]: ax.axis('off')
        fig.suptitle('Keyframe Extraction (Inter-Frame Difference)', color='white', fontsize=13)
        plt.tight_layout(); plt.show()

---
## Summary — Integration with TRACE

| IVP Technique | Role in TRACE |
|---|---|
| **Keyframe Extraction** | Populates `images` table — 114 keyframes from 25 IndEgo videos |
| **Edge Detection** | Frame complexity scoring — high edge density = candidate keyframe |
| **Optical Flow** | Motion detection — high flow magnitude marks active assembly intervals |
| **SIFT/ORB** | Cross-frame feature matching — tracks workpiece components across shots |
| **HOG** | Compact shape descriptor — fast workpiece classification baseline |
| **CLAHE** | Pre-processing for SigLIP 2 visual embeddings — normalises lighting |
| **Color Space** | HSV saturation used for glare/shadow detection in industrial frames |
| **Morphology** | Binary mask cleaning for workpiece segmentation |
| **Convolution** | Foundation of SigLIP 2 and BGE-M3 (all deep nets are convolutions) |